In [1]:
//code
import pandas as pd
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = pd.read_csv(r"C:\Users\Tanuja\Downloads\breast_cancer_wisconsin.csv")

df.columns = df.columns.astype(str).str.strip()
df = df.dropna(axis=1, how="all")
df = df.loc[:, ~df.columns.str.lower().str.startswith("unnamed")]

target_col = next(
    (col for col in df.columns
     if col.lower() in ["diagnosis", "class", "target", "label"]),
    None
)

if target_col is None:
    raise ValueError("Target column not found. Check dataset columns.")

id_cols = [col for col in df.columns if col.lower() in ["id", "index"]]
df = df.drop(columns=id_cols)

X = df.drop(columns=[target_col]).select_dtypes(include="number")
y = df[target_col].astype(str).str.strip()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model1 = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

start = time.perf_counter()
model1.fit(X_train, y_train)
time1 = time.perf_counter() - start

start = time.perf_counter()
pred1 = model1.predict(X_test)
predict_time1 = time.perf_counter() - start

n_components = min(X.shape[1], y.nunique() - 1)

model2 = Pipeline([
    ("scaler", StandardScaler()),
    ("lda", LinearDiscriminantAnalysis(n_components=n_components)),
    ("classifier", LogisticRegression(max_iter=1000))
])

start = time.perf_counter()
model2.fit(X_train, y_train)
time2 = time.perf_counter() - start

start = time.perf_counter()
pred2 = model2.predict(X_test)
predict_time2 = time.perf_counter() - start

accuracy1 = accuracy_score(y_test, pred1)
accuracy2 = accuracy_score(y_test, pred2)

print("Original Number of Features:", X.shape[1])
print("Reduced Number of Features:", n_components)

results = pd.DataFrame({
    "Model": ["Without LDA", "With LDA"],
    "Features": [X.shape[1], n_components],
    "Accuracy": [accuracy1, accuracy2],
    "Training Time": [time1, time2],
    "Prediction Time": [predict_time1, predict_time2]
})

print("\nModel Performance Comparison:")
print(results.to_string(index=False))
//code

Original Number of Features: 30
Reduced Number of Features: 1

Model Performance Comparison:
      Model  Features  Accuracy  Training Time  Prediction Time
Without LDA        30  0.982456       0.023403         0.002981
   With LDA         1  0.973684       0.048978         0.003249


In [ ]:
%pip install pandas scikit-learn
# 1. AIM

To make use of the dimensionality reduction process using Linear Discriminant Analysis (LDA) on a suitable high-dimensional dataset and observe the influence of feature reduction on model performance and computational efficiency.

# 2. DATASET USED

**Dataset Name:** Breast Cancer Wisconsin (Diagnostic) Dataset

**Dataset Path:** `C:\Users\Tanuja\Downloads\breast_cancer_wisconsin.csv`

The Breast Cancer Wisconsin dataset is used for classification and dimensionality reduction.

The standard dataset contains:
- 569 records.
- 30 numerical input features.
- 2 target classes.
- Target variable: Diagnosis.

The two target classes are:
- M – Malignant (cancerous).
- B – Benign (non-cancerous).

The features represent characteristics of cell nuclei obtained from digitized images of fine needle aspirate (FNA) samples of breast masses.

The dataset contains measurements such as radius, texture, perimeter, area, smoothness, compactness, concavity and symmetry.

**Why is this dataset used?**

The dataset contains 30 numerical features, making it suitable for demonstrating dimensionality reduction. It also contains two classes, which makes it appropriate for applying Linear Discriminant Analysis and observing its effect on classification performance.

# 3. THEORY

## A. Dimensionality Reduction

Dimensionality Reduction is the process of reducing the number of input features in a dataset while preserving important information.

High-dimensional datasets contain a large number of features, which can increase computational complexity, memory usage and model training time.

Dimensionality reduction transforms the original features into a smaller set of features to simplify machine learning operations.

**Need for Dimensionality Reduction:**

1. Reduces the number of input features.
2. Reduces computational complexity.
3. Reduces memory requirements.
4. Removes redundant information.
5. Helps improve model efficiency.
6. Can reduce overfitting in some situations.

## B. Linear Discriminant Analysis (LDA)

LDA stands for Linear Discriminant Analysis.

LDA is a supervised dimensionality reduction technique used to transform high-dimensional data into a lower-dimensional representation while maximizing the separation between different classes.

It uses both input features and target labels to find the most suitable projection directions.

LDA tries to:
- Maximize the distance between different classes.
- Minimize the variation within the same class.
- Preserve the information useful for class separation.

In this practical, LDA is used to reduce the original 30 features into a smaller number of discriminant components.

## C. Working of LDA

LDA works through the following steps:

1. Load the dataset containing input features and class labels.
2. Standardize the numerical input features.
3. Calculate the mean of each class.
4. Calculate within-class scatter and between-class scatter.
5. Find the directions that maximize class separation.
6. Select the required number of discriminant components.
7. Transform the original data into a lower-dimensional representation.

## D. Within-Class Scatter

Within-Class Scatter measures the variation of data points within the same class.

LDA tries to minimize within-class scatter so that data points belonging to the same class remain close together after projection.

## E. Between-Class Scatter

Between-Class Scatter measures the separation between different classes.

LDA tries to maximize between-class scatter so that different classes become well separated after projection.

## F. Mathematical Formula of LDA

The objective function of LDA is:

J(w) = (wᵀ Sᵦ w) / (wᵀ Sᵥ w)

Where:

- w = Projection direction.
- Sᵦ = Between-class scatter matrix.
- Sᵥ = Within-class scatter matrix.

LDA selects projection directions that maximize the ratio of between-class scatter to within-class scatter.

## G. Maximum Number of LDA Components

The maximum number of discriminant components depends on the number of input features and the number of classes.

Formula:

Maximum Components = min(Number of Features, Number of Classes - 1)

In this practical:

- Original features = 30.
- Number of classes = 2.

Therefore:

Maximum Components = min(30, 2 - 1)

Maximum Components = 1

Thus, LDA reduces the original 30 numerical features into one discriminant component.

## H. Standardization Before LDA

Standardization converts numerical features into a comparable scale.

It is performed using StandardScaler.

Formula:

Z = (X - μ) / σ

Where:

- X = Original value.
- μ = Mean of the feature.
- σ = Standard deviation.
- Z = Standardized value.

Standardization is used in this practical before applying LDA to prevent numerical differences in feature scales from unnecessarily affecting the analysis.

## I. Classification Before and After LDA

In this practical, Logistic Regression is used as the classification algorithm.

Two models are created:

**Model 1 – Without LDA:**

The Logistic Regression model uses all 30 original numerical features.

**Model 2 – With LDA:**

LDA reduces the original 30 features to one discriminant component. Logistic Regression then performs classification using this reduced feature.

The performance of both models is compared.

## J. Performance Evaluation

The following measurements are used to compare both models:

**1. Accuracy:**

Accuracy measures the proportion of correctly classified observations.

Accuracy = Correct Predictions / Total Predictions

**2. Training Time:**

Training Time measures the time required to train the complete model pipeline.

**3. Prediction Time:**

Prediction Time measures the time required to predict the output for the test dataset.

These measurements help us observe the influence of LDA on classification performance and computational efficiency.

## K. Advantages of LDA

1. Reduces the number of input features.
2. Improves class separation.
3. Reduces computational requirements for subsequent model operations.
4. Can help reduce overfitting.
5. Uses class labels to identify discriminative information.

## L. Limitations of LDA

1. LDA assumes certain statistical properties of the data, including class-conditional normality in its traditional formulation.
2. It assumes a common covariance structure for classes in its standard form.
3. It is a linear dimensionality reduction technique.
4. The maximum number of components depends on the number of classes.
5. Some information may be lost after dimensionality reduction.

# 4. ALGORITHM

1. Start.
2. Import the required Python libraries.
3. Load the Breast Cancer Wisconsin dataset using read_csv().
4. Remove unnecessary ID and empty columns.
5. Identify the target column.
6. Separate input features X and target variable y.
7. Divide the dataset into training and testing sets.
8. Create a Logistic Regression pipeline without LDA.
9. Train the first model using all original numerical features.
10. Calculate its training time and prediction time.
11. Apply StandardScaler and LDA to reduce the input features.
12. Train the second Logistic Regression model using reduced features.
13. Calculate its training time and prediction time.
14. Calculate the accuracy of both models.
15. Compare the number of features before and after LDA.
16. Display the model performance comparison.
17. Stop.

# 5. DETAILED EXPLANATION OF EACH LINE OF CODE

## A. Importing Libraries

**1. import pandas as pd**

Imports the Pandas library for loading, managing and manipulating datasets.

**2. import time**

Imports the time module to measure the training and prediction time of both models.

**3. from sklearn.model_selection import train_test_split**

Imports train_test_split() to divide the dataset into training and testing sets.

**4. from sklearn.preprocessing import StandardScaler**

Imports StandardScaler to standardize numerical features before applying LDA.

**5. from sklearn.discriminant_analysis import LinearDiscriminantAnalysis**

Imports the LinearDiscriminantAnalysis class used for dimensionality reduction.

**6. from sklearn.pipeline import Pipeline**

Imports Pipeline to combine preprocessing and classification steps into a single workflow.

**7. from sklearn.linear_model import LogisticRegression**

Imports Logistic Regression, which is used for classification.

**8. from sklearn.metrics import accuracy_score**

Imports accuracy_score() to calculate classification accuracy.

## B. Loading the Dataset

`df = pd.read_csv(r"C:\Users\Tanuja\Downloads\breast_cancer_wisconsin.csv")`

- read_csv() reads the CSV file.
- df stores the dataset in a DataFrame.
- The specified path identifies the location of the dataset.

## C. Cleaning Column Names

`df.columns = df.columns.astype(str).str.strip()`

Converts column names into strings and removes unnecessary spaces.

`df = df.dropna(axis=1, how="all")`

Removes columns containing only missing values.

`df = df.loc[:, ~df.columns.str.lower().str.startswith("unnamed")]`

Removes unwanted unnamed columns that may be created during CSV export.

## D. Identifying the Target Column

`target_col = next(...)`

Searches for the target column using common names such as diagnosis, class, target or label.

`if target_col is None:`

Checks whether a suitable target column was found.

`raise ValueError(...)`

Displays an error when the target column cannot be identified.

## E. Removing ID Columns

`id_cols = [col for col in df.columns if col.lower() in ["id", "index"]]`

Identifies columns used only for record identification.

`df = df.drop(columns=id_cols)`

Removes those unnecessary identification columns.

## F. Separating Input and Output Variables

`X = df.drop(columns=[target_col]).select_dtypes(include="number")`

Selects numerical input features and excludes the target column.

`y = df[target_col].astype(str).str.strip()`

Stores the target variable and converts its class labels into strings.

For this dataset:
- M represents Malignant.
- B represents Benign.

## G. Splitting the Dataset

`train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)`

Divides the dataset into training and testing sets.

- X = Input features.
- y = Target variable.
- test_size=0.2 = 20% testing data.
- random_state=42 = Reproducible splitting.
- stratify=y = Maintains class proportions in both sets.

## H. Model Without LDA

`model1 = Pipeline([...])`

Creates a pipeline with two steps:

1. StandardScaler standardizes the input features.
2. LogisticRegression performs classification.

`model1.fit(X_train, y_train)`

Trains the first model using all original numerical features.

`time.perf_counter()`

Records the starting and ending times to calculate training time.

`model1.predict(X_test)`

Predicts the class labels for the testing data.

## I. Selecting the Number of LDA Components

`n_components = min(X.shape[1], y.nunique() - 1)`

Calculates the maximum valid number of LDA components.

- X.shape[1] returns the number of input features.
- y.nunique() returns the number of unique classes.
- Subtracting 1 gives the maximum class-based component limit.

For 30 features and 2 classes, the result is 1.

## J. Model With LDA

`model2 = Pipeline([...])`

Creates a pipeline containing:

1. StandardScaler.
2. LinearDiscriminantAnalysis.
3. LogisticRegression.

`LinearDiscriminantAnalysis(n_components=n_components)`

Creates an LDA object that reduces the original features to the selected number of discriminant components.

`model2.fit(X_train, y_train)`

Fits the scaler, learns the LDA transformation using training features and class labels, and trains Logistic Regression using the reduced training data.

## K. Prediction After LDA

`model2.predict(X_test)`

Predicts the target classes using the reduced-dimensional testing data.

The same fitted scaling and LDA transformations learned from training data are applied automatically.

## L. Calculating Accuracy

`accuracy_score(y_test, pred1)`

Calculates the accuracy of the model without LDA.

`accuracy_score(y_test, pred2)`

Calculates the accuracy of the model with LDA.

## M. Comparing Results

`results = pd.DataFrame({...})`

Creates a comparison table containing:

- Model name.
- Number of features.
- Accuracy.
- Training time.
- Prediction time.

`results.to_string(index=False)`

Displays the comparison table without row indices.

This helps observe whether dimensionality reduction affects classification performance and computational efficiency.

# 6. EXPECTED OUTPUT

For the standard Breast Cancer Wisconsin Diagnostic dataset:

- Number of records: 569.
- Original numerical features: 30.
- Number of classes: 2.
- Maximum LDA components: 1.
- Training records: 455.
- Testing records: 114.

The program displays:

1. Original number of features.
2. Reduced number of features.
3. Accuracy without LDA.
4. Accuracy with LDA.
5. Training time without LDA.
6. Training time with LDA.
7. Prediction time without LDA.
8. Prediction time with LDA.

The accuracy and timing values are calculated during execution and should not be assumed in advance.

# 7. ORAL EXAMINATION – VIVA QUESTIONS AND ANSWERS

**Q1. What is Dimensionality Reduction?**

Dimensionality Reduction is the process of reducing the number of input features while preserving useful information.

**Q2. What is LDA?**

LDA stands for Linear Discriminant Analysis. It is a supervised dimensionality reduction technique that maximizes separation between different classes.

**Q3. Why is LDA used?**

LDA is used to reduce the number of features and improve separation between different classes.

**Q4. Is LDA supervised or unsupervised?**

LDA is a supervised technique because it uses class labels during dimensionality reduction.

**Q5. What is a discriminant component?**

A discriminant component is a new feature obtained by projecting the original data in a direction that maximizes class separation.

**Q6. What is within-class scatter?**

It measures the variation among data points belonging to the same class.

**Q7. What is between-class scatter?**

It measures the separation between different classes.

**Q8. What is the main objective of LDA?**

To maximize between-class scatter and minimize within-class scatter.

**Q9. What is StandardScaler?**

StandardScaler standardizes numerical attributes to approximately zero mean and unit standard deviation.

**Q10. What is Logistic Regression?**

Logistic Regression is a supervised machine learning algorithm used for classification.

**Q11. What is the purpose of train_test_split()?**

It divides the dataset into training and testing sets.

**Q12. What is accuracy?**

Accuracy is the ratio of correctly classified observations to the total number of observations.

**Q13. What is a feature?**

A feature is an input variable used by a machine learning model.

**Q14. What is a target variable?**

A target variable is the output that a supervised learning model tries to predict.

**Q15. What is the main difference between PCA and LDA?**

PCA maximizes overall data variance without using class labels, whereas LDA uses class labels to maximize class separation.

# 8. DIFFICULT EXTERNAL EXAMINER QUESTIONS AND ANSWERS

**Q16. Why is LDA called a supervised dimensionality reduction technique?**

Because it uses both input features and known class labels to find directions that best separate different classes.

**Q17. What is the maximum number of LDA components?**

The maximum number of LDA components is:

min(Number of Features, Number of Classes - 1).

**Q18. Why does LDA produce only one component in this practical?**

Because the dataset contains two classes. Therefore, the maximum number of components is 2 - 1 = 1.

**Q19. What happens if we have three classes?**

The maximum possible number of LDA components is two, provided there are at least two input features.

**Q20. What is the difference between PCA and LDA?**

PCA is unsupervised and maximizes overall variance, whereas LDA is supervised and maximizes class separability.

**Q21. Can LDA be used as a classifier?**

Yes. LDA can perform classification directly. In this practical, it is used as a dimensionality reduction step before Logistic Regression.

**Q22. Why is StandardScaler applied before LDA?**

It standardizes numerical features so that differences in their measurement scales do not unnecessarily influence the analysis.

**Q23. What happens if within-class scatter is singular?**

The within-class scatter matrix cannot be directly inverted using the ordinary inverse operation. Numerical methods or regularization techniques may be required.

**Q24. Does LDA always improve accuracy?**

No. LDA may discard information useful for classification. Therefore, accuracy may increase, decrease or remain similar.

**Q25. What is Data Leakage?**

Data Leakage occurs when information from testing data is used during model training or preprocessing.

**Q26. How is Data Leakage prevented in this practical?**

The dataset is split first, and the StandardScaler and LDA transformations are fitted using training data only through the Pipeline.

**Q27. What is the difference between feature selection and feature extraction?**

Feature selection chooses existing features, whereas feature extraction creates new features by transforming the original features.

**Q28. What are the assumptions of traditional LDA?**

Traditional LDA commonly assumes normally distributed features within each class and equal class covariance matrices.

**Q29. Why is LDA useful for classification problems?**

Because it identifies directions that improve separation between different target classes.

**Q30. Does reducing 30 features to one mean that only one original feature is selected?**

No. The resulting LDA component is a linear combination of the original 30 features, not one selected original feature.

# 9. DATASET-RELATED QUESTIONS

**Q1. What is the Breast Cancer Wisconsin dataset?**

It is a medical dataset containing measurements of cell nuclei obtained from digitized images of fine needle aspirate samples of breast masses.

**Q2. How many records are in the standard Diagnostic dataset?**

569 records.

**Q3. How many numerical features are present?**

30 numerical features.

**Q4. What are the two target classes?**

Malignant and Benign.

**Q5. What does Malignant mean?**

Malignant refers to a cancerous tumour.

**Q6. What does Benign mean?**

Benign refers to a non-cancerous tumour.

**Q7. What does the radius feature represent?**

It describes the distance from the centre of a cell nucleus to points on its boundary.

**Q8. What does the texture feature represent?**

It represents variation in grayscale values of the cell nucleus image.

**Q9. Why is this dataset suitable for LDA?**

It contains 30 numerical input features and two known classes, making it suitable for supervised dimensionality reduction and classification.

**Q10. What is the target column in this dataset?**

The target column is Diagnosis.

# 10. PRACTICAL EXAMINATION – CODING QUESTIONS

**Q1. How do you load the dataset?**

`pd.read_csv("breast_cancer_wisconsin.csv")`

**Q2. How do you select numerical features?**

`df.select_dtypes(include="number")`

**Q3. How do you divide the dataset?**

`train_test_split(X, y, test_size=0.2, stratify=y)`

**Q4. How do you standardize numerical data?**

`StandardScaler()`

**Q5. How do you apply LDA?**

`LinearDiscriminantAnalysis(n_components=1)`

**Q6. How do you train the model?**

`model.fit(X_train, y_train)`

**Q7. How do you predict output?**

`model.predict(X_test)`

**Q8. How do you calculate accuracy?**

`accuracy_score(y_test, y_pred)`

**Q9. How do you calculate execution time?**

`time.perf_counter()`

**Q10. How do you check the number of features?**

`X.shape[1]`

# 11. IMPORTANT FORMULAS

**1. LDA Objective Function**

J(w) = (wᵀ Sᵦ w) / (wᵀ Sᵥ w)

**2. Maximum LDA Components**

Maximum Components = min(Number of Features, Number of Classes - 1)

**3. Standardization**

Z = (X - μ) / σ

**4. Accuracy**

Accuracy = (Correct Predictions / Total Predictions) × 100

# 12. IMPORTANT POINTS FOR VIVA

1. LDA stands for Linear Discriminant Analysis.
2. LDA is a supervised dimensionality reduction technique.
3. LDA uses target class labels.
4. LDA maximizes between-class scatter and minimizes within-class scatter.
5. The maximum number of components is the minimum of the number of features and the number of classes minus one.
6. For two classes, LDA produces at most one component.
7. The Breast Cancer Wisconsin Diagnostic dataset contains 30 numerical features and two classes.
8. Logistic Regression is used for classification in this practical.
9. Model performance is compared using accuracy.
10. Computational efficiency is observed using training time and prediction time.